# R4 session (calibration or labeling) on one Qiskit base

**Edit only the first code cell** (which session this is). Then: **Settings**: Accelerator **None (CPU)**, **Internet OFF**, Environment **Pin to original environment**. **Save Version, Save & Run All (Commit)**.

Inputs to attach (Add Input, Datasets):
- `r4-code` (the registered version), always;
- the wheelhouse of this session's base: `r4-wheelhouse-qiskit-1-4-6`, `r4-wheelhouse-qiskit-2-3-1` or `r4-wheelhouse-qiskit-2-6-1` (all three are our own builds, made by the same pipeline as the fault overlays);
- labeling sessions only: the fault overlays of this base's pool (`r4-faults-c24`, `r4-faults-a231` or `r4-faults-b`).

A labeling session on a base runs only after that base's calibration session has finished. Resumable: if a session stops, attach its previous output as an input and run again.

In [ ]:
# ---- EDIT THIS CELL ONLY -------------------------------------------------------------
SESSION = "NR4a"          # NR4a NR4b NR4c (calibration) or R4-C1 R4-C2 R4-A R4-B1 R4-B2 (labeling)
# ----------------------------------------------------------------------------------------
PLAN = {   # session -> (mode, base, shard)
    "NR4a": ("calibrate", "2.6.1", None), "NR4b": ("calibrate", "1.4.6", None), "NR4c": ("calibrate", "2.3.1", None),
    "R4-C1": ("label", "1.4.6", "1/2"), "R4-C2": ("label", "1.4.6", "2/2"), "R4-A": ("label", "2.3.1", None),
    "R4-B1": ("label", "2.6.1", "1/2"), "R4-B2": ("label", "2.6.1", "2/2"),
}
MODE, BASE, SHARD = PLAN[SESSION]
print(SESSION, MODE, BASE, SHARD)

In [ ]:
import glob, json, os, shutil, subprocess, sys
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"not found: {pattern}  (is the dataset attached?)"
    return hits[0]
CODE = os.path.dirname(os.path.dirname(find_one("/kaggle/input/**/reveal/label.py")))
WH = os.path.dirname(find_one("/kaggle/input/**/wheelhouse/MANIFEST.json"))
MAN = json.load(open(os.path.join(WH, "MANIFEST.json")))
assert MAN["qiskit_requested"] == BASE and "source" in MAN.get("qiskit_source", ""), f"attach the wheelhouse of our {BASE} build"
REQ = next(p for p in (os.path.join(WH, "requirements.txt"), os.path.join(WH, "requirements_p3.txt")) if os.path.exists(p))
for k, v in {"RAYON_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "QISKIT_IN_PARALLEL": "TRUE",
             "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}.items():
    os.environ[k] = v
WORK = "/kaggle/working/r4"
if not os.path.exists(WORK):
    shutil.copytree(CODE, WORK)
SITE = "/kaggle/working/site"
if not os.path.exists(os.path.join(SITE, "qiskit")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH, "--target", SITE,
                    "-r", REQ], check=True)
PY = sys.executable
PYPATH = f"{SITE}:{WORK}:{WORK}/cart/cart/src:{WORK}/cart/obsaware/src"
print(subprocess.run([PY, "-c", "import qiskit, qiskit_aer, sys; print(sys.version.split()[0], qiskit.__version__, qiskit.__file__, qiskit_aer.__version__)"],
                     capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=PYPATH)).stdout)
print("wheelhouse import check:", MAN["import_check"])

In [ ]:
# Pin this kernel (and every child process) to one core, as in R3.
TASKSET = shutil.which("taskset")
if TASKSET:
    print(subprocess.run([TASKSET, "-cp", "0", str(os.getpid())], capture_output=True, text=True).stdout)
else:
    print("WARNING: taskset not available; running unpinned (recorded in the run log)")

In [ ]:
OUT = f"/kaggle/working/out_{SESSION}"
prev = sorted(glob.glob(f"/kaggle/input/**/out_{SESSION}/hardware.json", recursive=True))
if prev and not os.path.exists(OUT):
    shutil.copytree(os.path.dirname(prev[0]), OUT)
    print("resuming from", os.path.dirname(prev[0]))
cmd = ([TASKSET, "-c", "0"] if TASKSET else []) + [PY, "-u", "-m", "reveal.label", MODE, "--items", f"{WORK}/data/items.json",
                                                  "--out", OUT, "--work", "/kaggle/working/fault_sites"]
if MODE == "label":
    FAULTS = os.path.dirname(os.path.dirname(find_one("/kaggle/input/**/faults/*/fault.json")))
    cmd += ["--faults", FAULTS, "--site", SITE]
    if SHARD:
        cmd += ["--shard", SHARD]
env = dict(os.environ, PYTHONPATH=PYPATH, CART_ROOT=f"{WORK}/cart", REVEAL_DATA=f"{WORK}/data")
p = subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("exit code", p.wait())

In [ ]:
shutil.make_archive(f"/kaggle/working/out_{SESSION}", "zip", OUT)
print(os.listdir("/kaggle/working"))